In [1]:
import sys
import kagglehub
import os

print(sys.executable)
print(kagglehub.__version__)
!{sys.executable} -m pip install --upgrade certifi requests kagglehub

/home/dev_recluse/Repos/machine-learning/phishing-detection/.venv/bin/python
1.0.2


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix)

In [3]:
##Part 1: Explore the dataset
#To download dataset from Kaggel
path = kagglehub.dataset_download("ethancratchley/email-phishing-dataset")

#Find CSV files in the dataset
csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]

#Display available CSV files
print(csv_files)

['email_phishing_data.csv']


In [4]:
#To load the first CSV file into a python DataFrame
df = pd.read_csv(os.path.join(path, csv_files[0]))

#To preview the data
display("Header:",df.head())
print("\nShape: \n", df.shape)
print("\nInfo: \n", df.info())

'Header:'

,num_words,num_unique_words,num_stopwords,num_links,num_unique_domains,num_email_addresses,num_spelling_errors,num_urgent_keywords,label
0,140,94,52,0,0,0,0,0,0
1,5,5,1,0,0,0,0,0,0
2,34,32,15,0,0,0,0,0,0
3,6,6,2,0,0,0,0,0,0
4,9,9,2,0,0,0,0,0,0



Shape: 
 (524846, 9)
<class 'pandas.DataFrame'>
RangeIndex: 524846 entries, 0 to 524845
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   num_words            524846 non-null  int64
 1   num_unique_words     524846 non-null  int64
 2   num_stopwords        524846 non-null  int64
 3   num_links            524846 non-null  int64
 4   num_unique_domains   524846 non-null  int64
 5   num_email_addresses  524846 non-null  int64
 6   num_spelling_errors  524846 non-null  int64
 7   num_urgent_keywords  524846 non-null  int64
 8   label                524846 non-null  int64
dtypes: int64(9)
memory usage: 36.0 MB

Info: 
 None


In [5]:
#To examine the target variable
print(df["label"].value_counts())

label
0    517897
1      6949
Name: count, dtype: int64


In [6]:
display(df.groupby("label").first())

,num_words,num_unique_words,num_stopwords,num_links,num_unique_domains,num_email_addresses,num_spelling_errors,num_urgent_keywords
label,,,,,,,,
0,140,94,52,0,0,0,0,0
1,102,75,39,2,2,0,4,0


In [7]:
#To change to percentage values
print(df["label"].value_counts(normalize=True) * 100)

label
0    98.675993
1     1.324007
Name: proportion, dtype: float64


In [8]:
##Part 2: Prepare the data
#Separate the features from the target
X = df.drop("label", axis=1)
y = df["label"]

#Split the dataset into 80/20 training/testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

#Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
##Part 3: Train a Linear SVM
linear_svm = LinearSVC(C=1, class_weight="balanced", max_iter=10000, random_state=42)
linear_svm.fit(X_train_scaled, y_train)

#To use the training model to predict the test data
y_pred_linear = linear_svm.predict(X_test_scaled)

#To evaluate the model
print("\nAccuracy: ", accuracy_score(y_test, y_pred_linear))
print("\nClassification Report: \n", classification_report(y_test, y_pred_linear))
print("\nConfusion Matrix: \n", confusion_matrix(y_test, y_pred_linear))


Accuracy:  0.5365151948175669

Classification Report: 
               precision    recall  f1-score   support

           0       0.99      0.53      0.69    103580
           1       0.02      0.74      0.04      1390

    accuracy                           0.54    104970
   macro avg       0.51      0.64      0.37    104970
weighted avg       0.98      0.54      0.69    104970


Confusion Matrix: 
 [[55284 48296]
 [  356  1034]]


In [10]:
##Part 4: Linear vs RBF Kernel
#To train another SVM using RBF kernel
X_subset, _, y_subset, _ = train_test_split(X_train_scaled, y_train, train_size=30000, stratify=y_train, random_state=42)

rbf_svm = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", cache_size=4000)
rbf_svm.fit(X_subset, y_subset)             
y_pred_rbf = rbf_svm.predict(X_test_scaled)

#To evaluate the model
print("\nAccuracy: ", accuracy_score(y_test, y_pred_rbf))
print("\nClassification Report: \n", classification_report(y_test, y_pred_rbf))
print("\nConfusion Matrix: \n", confusion_matrix(y_test, y_pred_rbf))


Accuracy:  0.6371820520148614

Classification Report: 
               precision    recall  f1-score   support

           0       0.99      0.64      0.78    103580
           1       0.03      0.72      0.05      1390

    accuracy                           0.64    104970
   macro avg       0.51      0.68      0.41    104970
weighted avg       0.98      0.64      0.77    104970


Confusion Matrix: 
 [[65888 37692]
 [  393   997]]


In [ ]:
## Part 5: Experiment with the data
#Testing the values using the RBF kernel
for c in [0.01, 0.1, 1, 10, 100]:
    model = SVC(kernel="rbf", C=c)
    model.fit(X_train_scaled, y_train)
    test_accuracy = model.score(X_test_scaled, y_test)

    print("C = ", c, "Test Accuracy = ", test_accuracy)

C =  0.01 Test Accuracy =  0.9867581213680099
C =  0.1 Test Accuracy =  0.9867581213680099
C =  1 Test Accuracy =  0.9867581213680099
C =  10 Test Accuracy =  0.9867676478993999
